In [6]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping

# 1. Chargement et exploration des données
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/heart-disease.data"  # URL du dataset
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

# Charger les données
dataset = pd.read_csv(url, names=columns)

# Affichage des premières lignes du dataset
print(dataset.head())

# Visualisation des données
dataset.describe()

# 2. Prétraitement des données
# Gérer les valeurs manquantes (s'il y en a)
dataset = dataset.apply(lambda x: x.fillna(x.mean()) if x.dtype in ['float64', 'int64'] else x)

# Encodage des variables qualitatives (par exemple 'sex' et 'thal')
# Pour le cas de 'sex', on va simplement le mapper de façon binaire
dataset['sex'] = dataset['sex'].map({0: 'Female', 1: 'Male'})

# Sélectionner les colonnes numériques uniquement pour la normalisation
X = dataset.drop(columns=['target'])  # Exclure la colonne cible (target)
y = dataset['target']  # La cible reste inchangée

# Normalisation des données (uniquement sur les colonnes numériques)
scaler = StandardScaler()
X_scaled = X.select_dtypes(include=['float64', 'int64'])  # Sélectionner uniquement les colonnes numériques
X_scaled = scaler.fit_transform(X_scaled)  # Normaliser

# Convertir X_scaled de nouveau en DataFrame avec les mêmes colonnes
X[X_scaled.columns] = X_scaled

# 3. Division des données en ensembles d'entraînement, validation et test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.33, random_state=42)  # 0.33 pour avoir 10% du total

# Afficher les premières lignes après la normalisation
print(X_train.head())

# 4. Choix de l'architecture du modèle
model = Sequential()

# Ajouter les couches d'entrée et cachées
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.5))  # Dropout pour réduire l'overfitting
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))

# Couche de sortie avec activation sigmoïde pour la classification binaire
model.add(Dense(1, activation='sigmoid'))

# Résumé de l'architecture du modèle
model.summary()

# 5. Compilation du modèle
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 6. Entraînement du modèle
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  # Early stopping pour éviter le surapprentissage
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stopping])

# 7. Évaluation du modèle
# Prédictions
y_pred = (model.predict(X_test) > 0.5)

# Évaluation des performances
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# 8. Visualisation de l'évolution des pertes et de la précision au fil des époques
plt.figure(figsize=(12, 5))

# Perte d'entraînement vs validation
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Loss (Train)')
plt.plot(history.history['val_loss'], label='Loss (Validation)')
plt.title('Loss vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Précision d'entraînement vs validation
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Accuracy (Train)')
plt.plot(history.history['val_accuracy'], label='Accuracy (Validation)')
plt.title('Accuracy vs Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()



URLError: <urlopen error [Errno 11001] getaddrinfo failed>